# IR4 + Cloud 7B
Mount `ir4_7b_v1.zip` as a Kaggle Input and paste the local package command's `manifest_sha256` into the configuration cell. A CUDA GPU and internet access are required.

Before running, build the package locally and copy its printed `manifest_sha256` into `EXPECTED_MANIFEST_SHA256` in the configuration cell. Attach only that verified private ZIP or extracted package to Kaggle.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, re, shutil, subprocess, sys, zipfile
from huggingface_hub import HfApi

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
BUNDLE_INPUT = None  # ZIP or extracted directory containing bundle_manifest.json
EXPECTED_MANIFEST_SHA256 = None  # paste manifest_sha256 from the trusted package command
WEIGHTS_DIR = None  # optional directory containing cuhkx_weights.json


## 1. repo 与运行目录


In [ ]:
import hmac, stat

MAX_ARCHIVE_BYTES = 512 * 1024**2
MAX_ARCHIVE_FILES = 20_000
MAX_EXPANDED_BYTES = 512 * 1024**2
MAX_MEMBER_BYTES = 16 * 1024**2
MAX_COMPRESSION_RATIO = 50.0
FREE_SPACE_RESERVE = 256 * 1024**2

if not isinstance(EXPECTED_MANIFEST_SHA256, str) or re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST_SHA256) is None:
    raise RuntimeError("set EXPECTED_MANIFEST_SHA256 from the trusted local package command")


def _safe_bundle_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name
            or path.as_posix() != name or any(ord(character) < 32 for character in name)):
        raise RuntimeError("unsafe package path: " + repr(name))
    return path


def _open_bounded_archive(bundle):
    if not bundle.is_file():
        return None, None, 0
    if bundle.stat().st_size > MAX_ARCHIVE_BYTES:
        raise RuntimeError("package exceeds compressed-size budget")
    archive = zipfile.ZipFile(bundle)
    infos = archive.infolist()
    if not infos or len(infos) > MAX_ARCHIVE_FILES:
        raise RuntimeError("package entry count is outside the allowed budget")
    index, folded, expanded = {}, set(), 0
    for info in infos:
        _safe_bundle_name(info.filename)
        folded_name = info.filename.casefold()
        if info.filename in index or folded_name in folded:
            raise RuntimeError("package contains duplicate or case-colliding paths")
        mode = (info.external_attr >> 16) & 0xFFFF
        if (info.is_dir() or info.flag_bits & 1 or stat.S_IFMT(mode) not in (0, stat.S_IFREG)
                or info.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)):
            raise RuntimeError("package contains an unsupported entry")
        if info.file_size < 0 or info.file_size > MAX_MEMBER_BYTES:
            raise RuntimeError("package member exceeds size budget")
        if info.file_size and (not info.compress_size
                or info.file_size / info.compress_size > MAX_COMPRESSION_RATIO):
            raise RuntimeError("package member exceeds compression-ratio budget")
        expanded += info.file_size
        if expanded > MAX_EXPANDED_BYTES:
            raise RuntimeError("package exceeds expanded-size budget")
        index[info.filename] = info
        folded.add(folded_name)
    return archive, index, expanded


def _directory_member(bundle, name):
    source = bundle / name
    if source.is_symlink():
        raise RuntimeError("package symlinks are unsupported: " + name)
    path = source.resolve()
    if not path.is_relative_to(bundle.resolve()) or not path.is_file():
        raise RuntimeError("unsafe package file: " + name)
    if path.stat().st_size > MAX_MEMBER_BYTES:
        raise RuntimeError("package member exceeds size budget")
    return path


def _member_bytes(bundle, archive, index, name):
    if archive:
        info = index.get(name)
        if info is None:
            raise RuntimeError("package member is missing: " + name)
        with archive.open(info) as handle:
            content = handle.read(MAX_MEMBER_BYTES + 1)
        if len(content) != info.file_size or len(content) > MAX_MEMBER_BYTES:
            raise RuntimeError("package member size changed while reading")
        return content
    return _directory_member(bundle, name).read_bytes()


def _trusted_manifest(bundle, archive, index, marker):
    raw = _member_bytes(bundle, archive, index, marker)
    digest = hashlib.sha256(raw).hexdigest()
    if not hmac.compare_digest(digest, EXPECTED_MANIFEST_SHA256):
        raise RuntimeError("package manifest is not the trusted release")
    return raw, json.loads(raw)


def _validated_bundle_entries(bundle, archive, index, marker, manifest, prefix):
    entries = manifest.get("files")
    if not isinstance(entries, list) or len(entries) > MAX_ARCHIVE_FILES - 1:
        raise RuntimeError("invalid package file list")
    expected, folded, total = {}, set(), 0
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {"path", "bytes", "sha256"}:
            raise RuntimeError("invalid package entry")
        name, size, digest = entry["path"], entry["bytes"], entry["sha256"]
        path = _safe_bundle_name(name)
        if (not name.startswith(prefix) or not isinstance(size, int) or isinstance(size, bool)
                or size < 0 or size > MAX_MEMBER_BYTES or not isinstance(digest, str)
                or re.fullmatch(r"[0-9a-f]{64}", digest) is None):
            raise RuntimeError("invalid package path, size, or digest")
        if name in expected or name.casefold() in folded:
            raise RuntimeError("package manifest contains duplicate paths")
        if archive:
            if name not in index or index[name].file_size != size:
                raise RuntimeError("package manifest differs from ZIP metadata")
        else:
            if _directory_member(bundle, name).stat().st_size != size:
                raise RuntimeError("package manifest differs from directory metadata")
        expected[name] = entry
        folded.add(name.casefold())
        total += size
        if total > MAX_EXPANDED_BYTES:
            raise RuntimeError("package manifest exceeds expanded-size budget")
    actual = set(index) if archive else {marker, *expected}
    if actual != set(expected) | {marker}:
        raise RuntimeError("package file set differs from manifest")
    disk_root = WORK
    while not disk_root.exists():
        disk_root = disk_root.parent
    if shutil.disk_usage(disk_root).free < total + FREE_SPACE_RESERVE:
        raise RuntimeError("insufficient free space for bounded extraction")
    return list(expected.values())


def _verify_entry(bundle, archive, index, entry):
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    with source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            size += len(block)
            if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                raise RuntimeError("package member exceeded manifest size")
            digest.update(block)
    if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
        raise RuntimeError("package content hash mismatch: " + entry["path"])


def _copy_entry(bundle, archive, index, entry, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".partial")
    if partial.exists():
        partial.unlink()
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    try:
        with source, partial.open("xb") as output:
            for block in iter(lambda: source.read(1024 * 1024), b""):
                size += len(block)
                if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                    raise RuntimeError("package member exceeded manifest size")
                digest.update(block)
                output.write(block)
        if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
            raise RuntimeError("package content changed while copying")
        os.replace(partial, target)
    finally:
        if partial.exists():
            partial.unlink()

PACKAGE_ID = 'ir4_7b_v1'
MARKER = 'bundle_manifest.json'

if BUNDLE_INPUT is None:
    candidates = []
    for candidate_marker in INPUT.rglob(MARKER):
        try:
            if candidate_marker.stat().st_size <= MAX_MEMBER_BYTES:
                value = json.loads(candidate_marker.read_text(encoding="utf-8"))
                if value.get('baseline_id') == PACKAGE_ID:
                    candidates.append(candidate_marker.parent)
        except (OSError, ValueError):
            pass
    if not candidates:
        candidates = list(INPUT.rglob('ir4_7b_v1.zip'))
    if len(candidates) != 1:
        raise RuntimeError(f"found {len(candidates)} matching packages; set BUNDLE_INPUT")
    BUNDLE_INPUT = candidates[0]
BUNDLE_INPUT = Path(BUNDLE_INPUT).resolve()
archive, archive_index, expanded_bytes = _open_bounded_archive(BUNDLE_INPUT)
try:
    raw_manifest, manifest = _trusted_manifest(
        BUNDLE_INPUT, archive, archive_index, MARKER)
    if manifest.get("schema_version") != 1 or manifest.get('baseline_id') != PACKAGE_ID:
        raise RuntimeError("wrong package identity")
    pass
    entries = _validated_bundle_entries(
        BUNDLE_INPUT, archive, archive_index, MARKER, manifest, 'repo/')
    for entry in entries:
        _verify_entry(BUNDLE_INPUT, archive, archive_index, entry)
    MANIFEST_SHA256 = hashlib.sha256(raw_manifest).hexdigest()
    WORK_ROOT = WORK.resolve()
    RUNTIME = WORK_ROOT / ('ir4_source_' + MANIFEST_SHA256[:12])
    if RUNTIME.is_symlink():
        raise RuntimeError("runtime root must not be a symlink")
    RUNTIME_ROOT = RUNTIME.resolve()
    if not RUNTIME_ROOT.is_relative_to(WORK_ROOT):
        raise RuntimeError("runtime root escapes working directory")
    for entry in entries:
        candidate = RUNTIME_ROOT / entry["path"]
        target = candidate.resolve()
        if candidate.is_symlink() or not target.is_relative_to(RUNTIME_ROOT):
            raise RuntimeError("runtime path escapes package root")
        if target.exists():
            if target.is_symlink() or not target.is_file():
                raise RuntimeError("runtime contains an unsafe existing path")
            digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if target.stat().st_size != entry["bytes"] or digest != entry["sha256"]:
                raise RuntimeError("runtime copy was modified; use a new experiment/runtime")
    for entry in entries:
        target = (RUNTIME_ROOT / entry["path"]).resolve()
        if not target.exists():
            _copy_entry(BUNDLE_INPUT, archive, archive_index, entry, target)
    REPO = RUNTIME_ROOT / 'repo'
    (RUNTIME_ROOT / MARKER).write_bytes(raw_manifest)
finally:
    if archive:
        archive.close()
print("Verified repository:", REPO)
print("Trusted manifest SHA256:", MANIFEST_SHA256)
DEST = WORK / "ir4_runtime"
DEST.mkdir(parents=True, exist_ok=True)
PINNED_REVISION = HfApi().model_info("Qwen/Qwen2.5-VL-7B-Instruct").sha
print("revision:", PINNED_REVISION)
print("REPO:", REPO, "| DEST:", DEST)


## 2. 独立 Python 3.11 环境


In [ ]:
VENV = DEST / "venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    with tempfile.TemporaryDirectory(prefix="cuhkx_uv_bootstrap_", dir=WORK) as bootstrap_dir:
        BOOT = Path(bootstrap_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(BOOT), "--no-deps", "--require-hashes", "--only-binary=:all:", "-r", str(REPO / "requirements/bootstrap.lock.txt")], check=True)
        uv_env = {**os.environ, "PYTHONPATH": str(BOOT)}
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", str(VENV)], env=uv_env, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--require-hashes", "--only-binary=:all:", "-r", str(REPO / "requirements/cloud.lock.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
(DEST / "environment.freeze.txt").write_text(subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True))
probe = "import json,sys,torch; assert sys.version_info[:2]==(3,11); assert torch.cuda.is_available(),'需要 CUDA GPU'; print(json.dumps({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))"
environment = subprocess.check_output([str(PYTHON), "-c", probe], text=True)
(DEST / "environment.json").write_text(environment)
print(environment)

def cloud(*args): subprocess.run([str(PYTHON), "-m", "cuhkx.cli", *args, "--project-root", str(REPO)], cwd=REPO, check=True)


## 3. 锁定 revision、准备权重和检查缓存


In [ ]:
if not re.fullmatch(r"[0-9a-f]{40}", PINNED_REVISION): raise RuntimeError("无效的 40 位 revision")
pin_code = "import sys,yaml; from pathlib import Path; p=Path(sys.argv[1]); v=yaml.safe_load(p.read_text()); old=v['model']['revision']; assert old is None or old==sys.argv[2], '已锁定不同 revision'; v['model']['revision']=sys.argv[2]; p.write_text(yaml.safe_dump(v,sort_keys=False))"
subprocess.run([str(PYTHON), "-c", pin_code, str(REPO / "configs/baseline.yaml"), PINNED_REVISION], check=True)

WEIGHTS_DIR = Path(WEIGHTS_DIR) if WEIGHTS_DIR else next((p.parent for p in INPUT.rglob("cuhkx_weights.json")), None)
if WEIGHTS_DIR is None:
    WEIGHTS_DIR = Path("/tmp") / f"ir4_weights_{PINNED_REVISION[:12]}"
    if shutil.disk_usage("/tmp").free < 18 * 1024**3: raise RuntimeError("没有足够空间下载 7B 权重；请挂载含 cuhkx_weights.json 的已验证权重数据集")
print("WEIGHTS_DIR:", WEIGHTS_DIR)
cloud("fetch-weights", "--weights-dir", str(WEIGHTS_DIR))
cloud("check", "--dataset", "pilot")
cloud("check", "--dataset", "test")


## 4. Smoke：固定 pilot 前 16 QA


In [ ]:
cloud("predict", "--dataset", "pilot", "--limit", "16", "--run-id", "ir4_7b_smoke", "--weights-dir", str(WEIGHTS_DIR), "--resume")


## 5. Pilot：120 QA 与独立评分（不设置准确率阈值）


In [ ]:
cloud("verify-run", "--run-id", "ir4_7b_smoke")
cloud("predict", "--dataset", "pilot", "--run-id", "ir4_7b_pilot", "--weights-dir", str(WEIGHTS_DIR), "--resume")
cloud("evaluate", "--run-id", "ir4_7b_pilot")


## 6. Test：682 QA 与本地提交文件导出


In [ ]:
for run_id, count in (("ir4_7b_smoke", 16), ("ir4_7b_pilot", 120)):
    verified = json.loads(subprocess.check_output([str(PYTHON), "-m", "cuhkx.cli", "verify-run", "--run-id", run_id, "--project-root", str(REPO)], text=True))
    if (verified["dataset"], verified["target_qa"], verified["execution_mode"]) != ("pilot", count, "cloud"): raise RuntimeError("前置阶段验证失败")
cloud("predict", "--dataset", "test", "--run-id", "ir4_7b_test", "--weights-dir", str(WEIGHTS_DIR), "--resume")
cloud("submit", "--run-id", "ir4_7b_test")
print("提交文件:", REPO / "outputs/ir4_7b_test/submission.csv")
print("运行证据:", REPO / "outputs", "| 环境证据:", DEST)
